# [PBT] 화장품 이커머스 ㅣ 05. 목표변수 생성

## 이 노트북이 하는 일

03번 노트북(배지환)이 만든 **대상 테이블**을 받아 목표변수 `purchase_7d`를 붙인다.

| 조각 | 내용 | 담당 |
|---|---|---|
| ① | 대상 고객과 t₀ 확정 | 배지환 — `03번` |
| **②** | **목표변수 Y 생성** | **이슬기 — 이 노트북** |
| ③ | 설명변수 X — 조회 변수 4종 | 배지환 |

### 목표변수 정의

> t₀부터 7일 안에 **같은 고객이 담은 그 상품을** 산 기록이 있으면 `purchase_7d = 1`

```
       ←─── 30분 ───→ │  t₀  │ ←────────── 7일 ──────────→
        X를 만드는 구간   담기      Y를 판정하는 구간
```

t₀ 기준 왼쪽이 설명변수, 오른쪽이 목표변수다. 두 구간은 겹치지 않는다.

> **경계 규칙** — Y는 `t₀ < 구매시각 ≤ t₀ + 7일` 로 **같은 초를 제외**한다.
> Y의 기준은 "담아둔 뒤 구매로 이어졌는가"이므로, 담자마자 산 즉시 구매는 이 질문의 범위 밖이다.
> 해당 건수는 `#04-4`에서 세어 결정기록에 남긴다.
> (X의 30분 창은 반대로 t₀를 **포함**한다. 기준이 "담는 시점에 알 수 있었는가"로 다르기 때문이다.)

## #01. 준비작업

### 1. 라이브러리 참조

In [1]:
import os, sys

# 프로젝트 폴더 — notebooks/ 안에서 실행하면 상위 폴더를 가리킨다
from pathlib import Path
BASE = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
os.chdir(BASE)             # 폰트 경로가 상대경로라 여기서 실행해야 한다
sys.path.insert(0, BASE)

from pandas import read_csv, read_parquet, to_datetime, Timedelta, DataFrame, set_option

from helpers import my_qtcheck

set_option("display.max_columns", 50)

### 2. 경로 지정

- 프로젝트 파일은 모두 한 폴더 아래에 있다. 원본 로그도 대상 테이블도 `output` 안이다.
- 각자 PC 경로가 다르므로 **아래 한 줄만** 자기 것으로 바꾼다.


In [2]:
OUT  = BASE + r"\output"


### 3. 대상 테이블 불러오기

- CSV는 **타입을 보존하지 않는다.** 01번에서 정한 타입을 여기서 되살린다.

| 컬럼 | CSV에서 읽으면 | 되살릴 타입 | 근거 |
|---|---|---|---|
| `t0` | 문자열 | datetime | 시간 계산을 해야 한다 |
| `product_id` | int64 | **int32** | 01번 `#03-6`에서 정한 타입 |
| `brand` · `category_id` | 문자열 / int64 | category | 명목형이다 |

- `utf-8-sig`로 저장된 파일이므로 같은 인코딩으로 읽는다.


In [3]:
target = read_csv(OUT + r"\02a_target.csv", encoding="utf-8-sig")

target["t0"]         = to_datetime(target["t0"], format="%Y-%m-%d %H:%M:%S")
target["product_id"] = target["product_id"].astype("int32")

target = my_qtcheck.set_type(target, as_category=["brand", "category_id"])


<class 'pandas.DataFrame'>
RangeIndex: 108458 entries, 0 to 108457
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   user_id      108458 non-null  int64         
 1   t0           108458 non-null  datetime64[us]
 2   product_id   108458 non-null  int32         
 3   price        108458 non-null  float64       
 4   brand        66322 non-null   category      
 5   category_id  108458 non-null  category      
dtypes: category(2), datetime64[us](1), float64(1), int32(1), int64(1)
memory usage: 3.3 MB


### 4. 대상 테이블 확인

- 1행 = 1고객인지 먼저 본다. 이후 모든 작업이 이 전제 위에 있다.

In [4]:
print(f"행 수    : {len(target):,}")
print(f"고객 수  : {target['user_id'].nunique():,}")
print(f"1행=1고객: {len(target) == target['user_id'].nunique()}")
print(f"t0 범위  : {target['t0'].min()}  ~  {target['t0'].max()}")

행 수    : 108,458
고객 수  : 108,458
1행=1고객: True
t0 범위  : 2019-10-11 00:00:17  ~  2019-11-24 02:58:43


### 5. 원본 로그에서 필요한 컬럼만 불러오기

- Y를 만드는 데 필요한 것은 **누가 · 무엇을 · 언제 샀는가** 세 가지뿐이다.
- 9개 컬럼을 다 읽으면 800만 행이 메모리를 크게 차지한다. **네 개만 읽는다.**

In [5]:
log = read_parquet(OUT + r"\01_qtcheck.parquet",
                   columns=["user_id", "product_id", "event_time", "event_type"])

print(f"불러온 행 수 : {len(log):,}")
log.head()

불러온 행 수 : 8,259,159


,user_id,product_id,event_time,event_type
0,463240011,5773203,2019-10-01 03:00:00,cart
1,463240011,5773353,2019-10-01 03:00:03,cart
2,429681830,5881589,2019-10-01 03:00:07,cart
3,463240011,5723490,2019-10-01 03:00:07,cart
4,429681830,5881449,2019-10-01 03:00:15,cart


## #02. `purchase` 이벤트 추출

### 1. 구매 기록만 남긴다

- 뒤에서 담기 시각(`t0`)과 구별해야 하므로 컬럼명을 `buy_time`으로 바꾼다.

In [6]:
buys = log.loc[log["event_type"] == "purchase",
               ["user_id", "product_id", "event_time"]]

buys = buys.rename(columns={"event_time": "buy_time"})

print(f"구매 이벤트 : {len(buys):,}건")
print(f"구매한 고객 : {buys['user_id'].nunique():,}명")
buys.head()

구매 이벤트 : 567,514건
구매한 고객 : 52,523명


,user_id,product_id,buy_time
184,474232307,5619862,2019-10-01 03:06:35
186,474232307,5619866,2019-10-01 03:06:35
187,474232307,5696184,2019-10-01 03:06:35
188,474232307,5706784,2019-10-01 03:06:35
189,474232307,5723503,2019-10-01 03:06:35


### 2. 원본 로그 삭제

- 구매 기록을 뽑았으므로 800만 행은 더 필요 없다. 메모리를 돌려준다.

In [7]:
del log

import gc
gc.collect()

60

## #03. 대상 테이블과 병합

### 1. 키가 두 개인 이유

목표변수가 **"담은 그 상품을"** 샀는지이므로, `user_id` 하나로 병합하면 안 된다.

| 병합 키 | 붙는 것 | 문제 |
|---|---|---|
| `user_id` 하나 | 그 고객이 산 **모든 상품** | 담았던 그 상품인지 구분할 수 없다 |
| `user_id` + `product_id` | 담은 그 상품의 구매만 | — |

A를 담고 B를 산 고객은 이 병합에서 아무것도 붙지 않는다. 의도한 결과다.

In [8]:
KEY = ["user_id", "product_id"]

m = target[["user_id", "product_id", "t0"]].merge(buys, on=KEY, how="inner")

print(f"담은 그 상품을 산 기록 : {len(m):,}건")
print(f"해당 고객 수           : {m['user_id'].nunique():,}명")
m.head()

담은 그 상품을 산 기록 : 25,003건
해당 고객 수           : 24,111명


,user_id,product_id,t0,buy_time
0,385999154,5753492,2019-10-11 00:01:51,2019-10-11 00:23:10
1,385999154,5753492,2019-10-11 00:01:51,2019-10-14 12:06:21
2,352064162,5635081,2019-10-11 00:05:15,2019-10-12 11:41:33
3,502631750,5670733,2019-10-11 00:07:07,2019-11-10 00:11:02
4,558945171,5652774,2019-10-11 00:09:00,2019-10-11 00:53:40


> 여기까지는 **기간을 따지지 않은** 전체 구매다. 담기 전에 산 것도 섞여 있다.
> 7일 창은 다음 장에서 적용한다.

## #04. 7일 창 적용

### 1. 시차 계산

- `buy_time − t0`가 양수면 담은 뒤 구매, 음수면 담기 전 구매다.

In [9]:
m["lag"] = m["buy_time"] - m["t0"]

m[["t0", "buy_time", "lag"]].head()

,t0,buy_time,lag
0,2019-10-11 00:01:51,2019-10-11 00:23:10,0 days 00:21:19
1,2019-10-11 00:01:51,2019-10-14 12:06:21,3 days 12:04:30
2,2019-10-11 00:05:15,2019-10-12 11:41:33,1 days 11:36:18
3,2019-10-11 00:07:07,2019-11-10 00:11:02,30 days 00:03:55
4,2019-10-11 00:09:00,2019-10-11 00:53:40,0 days 00:44:40


### 2. 시차의 분포부터 본다

- 창을 적용하기 전에 어떤 값들이 있는지 확인한다.

In [10]:
print(f"음수 (담기 전 구매)  : {(m['lag'] < Timedelta(0)).sum():,}건")
print(f"0    (같은 초 구매)  : {(m['lag'] == Timedelta(0)).sum():,}건")
print(f"양수 (담은 뒤 구매)  : {(m['lag'] > Timedelta(0)).sum():,}건")
print(f"7일 초과             : {(m['lag'] > Timedelta(days=7)).sum():,}건")

음수 (담기 전 구매)  : 11건
0    (같은 초 구매)  : 0건
양수 (담은 뒤 구매)  : 24,992건
7일 초과             : 2,267건


### 3. 같은 초 구매 — 결정기록 6번에 남길 값

- 담자마자 산 건이다. Y의 정의에서 제외하기로 했으므로 몇 건인지 기록한다.

In [11]:
same_sec = (m["lag"] == Timedelta(0)).sum()
same_user = m.loc[m["lag"] == Timedelta(0), "user_id"].nunique()

print(f"구매시각 == t0 : {same_sec:,}건  /  고객 {same_user:,}명")
print(f"전체 대상 대비 : {same_user / len(target):.3%}")

구매시각 == t0 : 0건  /  고객 0명
전체 대상 대비 : 0.000%


### 4. 창 안의 구매만 남긴다

- 조건은 `t₀ < 구매시각 ≤ t₀ + 7일` 이다.
- 왼쪽은 **초과**(같은 초 제외), 오른쪽은 **이하**(7일째 되는 순간까지 포함).

In [12]:
win = m.loc[(m["lag"] > Timedelta(0)) &
            (m["lag"] <= Timedelta(days=7))]

print(f"7일 창 안의 구매 : {len(win):,}건")
print(f"해당 고객 수     : {win['user_id'].nunique():,}명")

7일 창 안의 구매 : 22,725건
해당 고객 수     : 22,236명


## #05. 목표변수 생성

### 1. 구매한 고객 목록

- 한 고객이 같은 상품을 여러 번 살 수 있어 `win`에는 한 사람의 행이 여럿 남을 수 있다.
- 필요한 것은 **구매 여부뿐**이므로 목록으로 만든다.

In [13]:
buyers = win["user_id"].unique()

print(f"7일 내 구매한 고객 : {len(buyers):,}명")

7일 내 구매한 고객 : 22,236명


### 2. Y 부여

- **병합하지 않고 `isin`으로 판정한다.** 병합하면 행이 늘어나 1행 = 1고객이 깨진다.

In [14]:
target["purchase_7d"] = target["user_id"].isin(buyers).astype("int8")

target[["user_id", "t0", "product_id", "purchase_7d"]].head()

,user_id,t0,product_id,purchase_7d
0,543079411,2019-10-11 00:00:17,5591314,0
1,555391563,2019-10-11 00:01:47,5816172,0
2,385999154,2019-10-11 00:01:51,5753492,1
3,558950685,2019-10-11 00:02:04,5875777,0
4,558519615,2019-10-11 00:02:17,5810157,0


### 3. 행 수가 변하지 않았는지 확인

- 이 단계에서 행이 늘어나면 `isin`이 아니라 병합을 쓴 것이다.

In [15]:
print(f"행 수    : {len(target):,}")
print(f"고객 수  : {target['user_id'].nunique():,}")
print(f"1행=1고객: {len(target) == target['user_id'].nunique()}")

행 수    : 108,458
고객 수  : 108,458
1행=1고객: True


### 4. 첫 구매까지의 시차를 함께 남긴다

- 구매한 고객은 7일 중 **언제** 샀는지가 해석에 필요하다.
- 같은 상품을 여러 번 산 경우 **가장 이른 구매**를 쓴다.
- 미구매 고객은 값이 없으므로 결측이 된다. 이 결측은 "구매하지 않았다"는 뜻이다.
- 'lag_hours'는 담기로부터 구매까지 얼마나 걸렸는지(1시간 단위)를 뜻하는 변수이다.

In [16]:
first_lag = win.groupby("user_id")["lag"].min()

target["lag_hours"] = target["user_id"].map(first_lag / Timedelta(hours=1))

target[["user_id", "purchase_7d", "lag_hours"]].head()

,user_id,purchase_7d,lag_hours
0,543079411,0,NaN
1,555391563,0,NaN
2,385999154,1,0.355278
3,558950685,0,NaN
4,558519615,0,NaN


## #06. 양성 비율과 주 지표 확정

### 왜 여기서 하는가

계획 보고서는 Y 분포를 모르는 상태에서 주 지표를 ROC-AUC로 확정해 두었고,
피드백에서 **순서가 뒤바뀌었다**는 지적을 받았다.

클래스 불균형 정도를 모른 채 평가 지표를 고정해서는 안 된다.
**양성이 적으면 ROC-AUC는 실제보다 낙관적으로 나타난다.**
음성이 압도적으로 많을 경우 대부분을 0으로 예측해도 곡선이 양호하게 보이기 때문이다.

### 1. 양성 비율

In [17]:
pos_n = int(target["purchase_7d"].sum())
pos_ratio = target["purchase_7d"].mean()

dist = target["purchase_7d"].value_counts().sort_index().rename("고객 수").to_frame()
dist["비율 %"] = (dist["고객 수"] / len(target) * 100).round(2)
dist

,고객 수,비율 %
purchase_7d,,
0,86222,79.5
1,22236,20.5


### 2. 주 지표 판정

| 양성 비율 | 주 지표 | 보조 지표 |
|---|---|---|
| **10% 이상** | ROC-AUC | F1 · 정밀도 · 재현율 |
| **10% 미만** | PR-AUC (평균정밀도) | ROC-AUC · F1 · 정밀도 · 재현율 |

In [18]:
if pos_ratio >= 0.10:
    main_metric = "ROC-AUC"
    sub_metric  = "F1 · 정밀도 · 재현율"
else:
    main_metric = "PR-AUC (평균정밀도)"
    sub_metric  = "ROC-AUC · F1 · 정밀도 · 재현율"

print(f"양성 : {pos_n:,}명 / {len(target):,}명 = {pos_ratio:.2%}")
print()
print(f"주 지표   : {main_metric}")
print(f"보조 지표 : {sub_metric}")

양성 : 22,236명 / 108,458명 = 20.50%

주 지표   : ROC-AUC
보조 지표 : F1 · 정밀도 · 재현율


> **확정 — 이후 변경하지 않는다.**
> Y 분포를 확인한 직후 한 번 정하는 것은 정당하나, 모델 성능을 본 뒤 바꾸는 것은 허용되지 않는다.
> 위 결과를 **결정기록 6번**에 옮긴다.

## #07. 검증

### 1. 시차가 정의한 범위 안에 있는가

- 음수나 7일 초과가 하나라도 있으면 창 적용이 잘못된 것이다.

In [19]:
lag_ok = target.loc[target["purchase_7d"] == 1, "lag_hours"]

print(f"구매 고객 수     : {len(lag_ok):,}")
print(f"시차 최솟값(시간) : {lag_ok.min():.4f}")
print(f"시차 최댓값(시간) : {lag_ok.max():.4f}   (7일 = 168시간)")
print(f"범위 밖          : {((lag_ok <= 0) | (lag_ok > 168)).sum()}건")

구매 고객 수     : 22,236
시차 최솟값(시간) : 0.0039
시차 최댓값(시간) : 167.9747   (7일 = 168시간)
범위 밖          : 0건


### 2. 미구매 고객에게 시차가 없는가

In [20]:
no_buy = target.loc[target["purchase_7d"] == 0, "lag_hours"]

print(f"미구매 고객 수      : {len(no_buy):,}")
print(f"그중 시차가 있는 건 : {no_buy.notna().sum()}건   (0이어야 정상)")

미구매 고객 수      : 86,222
그중 시차가 있는 건 : 0건   (0이어야 정상)


### 3. 구매 시차 분포

- 담고 나서 얼마 만에 사는지 본다. 상식적인 모양인지 확인하는 목적이다.

In [21]:
step = DataFrame({
    "구간": ["1시간 내", "6시간 내", "1일 내", "3일 내", "7일 내"],
    "누적 비율 %": [
        (lag_ok <=   1).mean() * 100,
        (lag_ok <=   6).mean() * 100,
        (lag_ok <=  24).mean() * 100,
        (lag_ok <=  72).mean() * 100,
        (lag_ok <= 168).mean() * 100,
    ],
})

step["누적 비율 %"] = step["누적 비율 %"].round(1)
step

,구간,누적 비율 %
0,1시간 내,63.6
1,6시간 내,79.0
2,1일 내,87.2
3,3일 내,94.6
4,7일 내,100.0


### 4. 무작위 3명을 원본 로그와 대조

- 코드로 재계산하지 않고 **원본 기록을 눈으로 따라간다.**
- 확인할 것 — t₀ 이후 7일 안에 그 상품을 산 기록이 있으면 Y=1, 없으면 0이어야 한다.

In [22]:
sample_ids = target["user_id"].sample(3, random_state=3217).to_list()

for uid in sample_ids:
    row = target.loc[target["user_id"] == uid]
    pid = row["product_id"].iloc[0]

    print("=" * 72)
    print("user_id :", uid, " / product_id :", pid)
    print("[테이블]")
    print(row[["t0", "product_id", "purchase_7d", "lag_hours"]].to_string(index=False))
    print("[원본 로그 — 이 고객이 이 상품을 산 기록]")
    print(buys.loc[(buys["user_id"] == uid) & (buys["product_id"] == pid)]
              .sort_values("buy_time")
              .to_string(index=False))
    print()

user_id : 250311201  / product_id : 5671878
[테이블]
                 t0  product_id  purchase_7d  lag_hours
2019-11-03 10:03:59     5671878            1 116.503611
[원본 로그 — 이 고객이 이 상품을 산 기록]
  user_id  product_id            buy_time
250311201     5671878 2019-11-08 06:34:12

user_id : 565763975  / product_id : 5799891
[테이블]
                 t0  product_id  purchase_7d  lag_hours
2019-10-30 15:34:21     5799891            0        NaN
[원본 로그 — 이 고객이 이 상품을 산 기록]
Empty DataFrame
Columns: [user_id, product_id, buy_time]
Index: []

user_id : 560978725  / product_id : 5751383
[테이블]
                 t0  product_id  purchase_7d  lag_hours
2019-10-16 22:00:27     5751383            0        NaN
[원본 로그 — 이 고객이 이 상품을 산 기록]
Empty DataFrame
Columns: [user_id, product_id, buy_time]
Index: []



## #08. 저장

- 배지환의 ③(X 파생) 결과와 합쳐 마스터 테이블을 만들 때 쓴다.
- 19만 행이라 CSV로 둔다. 엑셀로 열어 눈으로 확인할 수 있다.
- 파일명은 `02c_y.csv` (04번 X 진단·이후 노트북이 이 이름으로 읽는다).

In [23]:
target.to_csv(OUT + r"\02c_y.csv", index=False, encoding="utf-8-sig")

print("저장 완료 :", OUT + r"\02c_y.csv")
print(f"{len(target):,}행 × {target.shape[1]}열")
target.dtypes

저장 완료 : .\output\02c_y.csv
108,458행 × 8열


user_id                 int64
t0             datetime64[us]
product_id              int32
price                 float64
brand                category
category_id          category
purchase_7d              int8
lag_hours             float64
dtype: object

In [ ]:
import numpy as np, matplotlib as mpl, matplotlib.pyplot as plt

mpl.rcParams['font.family'] = ['Malgun Gothic', 'AppleGothic', 'DejaVu Sans']
mpl.rcParams['axes.unicode_minus'] = False

# ── 7일 내 구매자 22,236명의 담기→구매 소요 시간(시간 단위) ──
lag = np.sort(target.loc[target['purchase_7d'] == 1, 'lag_hours'].to_numpy())
n = len(lag)

BLUE, RED, MUTE, GRID = '#2563EB', '#DC2626', '#6b7280', '#e5e7eb'
HOUR = {'1분': 1/60, '10분': 1/6, '1시간': 1, '6시간': 6,
        '1일': 24, '3일': 72, '7일': 168}

fig, ax = plt.subplots(figsize=(9.5, 4.2), dpi=200)
fig.patch.set_facecolor('white'); ax.set_facecolor('white')

# 누적 곡선 — 각 관측치가 자기 순위를 그대로 쓰므로 보간이 없다
ax.plot(lag, np.arange(1, n + 1) / n * 100, lw=2.4, color=BLUE, zorder=3)

# 눈금을 지수 표기가 아니라 실제 시간으로 바꾼다
ax.set_xscale('log')
ax.set_xticks(list(HOUR.values()))
ax.set_xticklabels(list(HOUR.keys()), fontsize=10, color=MUTE)
ax.minorticks_off()

# 목표변수의 경계선
ax.axvline(168, color=MUTE, ls=(0, (4, 3)), lw=1.2, zorder=2)
ax.text(168, 4, ' 7일 = 관측 창 끝', fontsize=10, color=MUTE, va='bottom')

# 주요 지점만 표시 — 값은 데이터에서 직접 읽는다
for label in ('1시간', '6시간', '1일', '3일'):
    t = HOUR[label]
    pct = (lag <= t).mean() * 100
    ax.plot(t, pct, 'o', ms=8, color=RED, mec='white', mew=2, zorder=5)
    ax.annotate(f'{label} {pct:.2f}%', xy=(t, pct), xytext=(9, -14),
                textcoords='offset points', fontsize=10.5,
                color=RED, fontweight='bold', zorder=6)

ax.set_ylim(0, 103); ax.set_xlim(1/60, 210)
ax.set_yticks([0, 25, 50, 75, 100])
ax.set_yticklabels(['0', '25', '50', '75', '100%'], fontsize=10, color=MUTE)
ax.set_xlabel('담은 뒤 경과 시간 (로그 눈금)', fontsize=10.5, color=MUTE, labelpad=6)
ax.set_ylabel('누적 비율', fontsize=10.5, color=MUTE, labelpad=6)

ax.grid(axis='y', color=GRID, lw=0.9, zorder=0)
for s in ('top', 'right', 'left'):
    ax.spines[s].set_visible(False)
ax.spines['bottom'].set_color('#d1d5db')

plt.tight_layout()
plt.savefig('fig03_시차누적.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

# ── 제가 렌더할 경우를 위한 출력 ──
g = np.logspace(np.log10(1/60), np.log10(168), 60)
print([round(x, 4) for x in g])
print([round((lag <= t).mean() * 100, 3) for t in g])